# DSA5206: three-model prediction-only error analysis
This independent notebook reads actual held-out predictions. It never loads model weights, extracts features, fits regressions or changes experimental outputs. All results below require a successfully validated real prediction CSV; there are no stored numerical findings or example clinical figures.

**Verified input filename:** the existing `EVA_X_Comparison.ipynb` exports `three_model_heldout_predictions.csv`, not `all_three_model_heldout_predictions.csv`. We inspect the export code again at runtime rather than inventing a fallback or deriving predictions from summary means. Default input folder: `/content/duong-eva-x-results/`.

Read `ERROR_ANALYSIS.md` for Colab setup and privacy instructions. All six figures use unique-image or equal-image patient summaries; repeated held-out predictions are not treated as independent X-rays.


In [ ]:
# Prediction-only dependencies; no torch, timm or checkpoints needed for analysis.
%pip install numpy==1.26.4 pandas==2.2.3 matplotlib==3.9.2


In [ ]:
# Optional: mount Drive if previous outputs are stored there.
# from google.colab import drive
# drive.mount('/content/drive')
import os,json
from pathlib import Path
import pandas as pd
from IPython.display import display,Image
REPO_DIR=Path(os.environ.get('DUONG_REPO_DIR','/content/nus_DSA5206_GROUPPROJECT')).resolve()
INPUT_DIR=Path(os.environ.get('DUONG_EVA_RESULTS_DIR','/content/duong-eva-x-results')).resolve()
OUTPUT_DIR=Path(os.environ.get('DUONG_ERROR_RESULTS_DIR','/content/duong-error-analysis-results')).resolve()
import sys
sys.path.insert(0,str(REPO_DIR))
from error_analysis_methods import (verified_prediction_filename,validate_predictions,analyze_predictions,
    presentation_table,export_analysis,FIGURE_NAMES,file_sha256)
source_notebook=REPO_DIR/'EVA_X_Comparison.ipynb'
verified_name=verified_prediction_filename(source_notebook)
PREDICTIONS=INPUT_DIR/verified_name
AUDIT=INPUT_DIR/'patient_split_audit.json'
MANIFEST=INPUT_DIR/'reproducibility_manifest.json'
print('Verified source export:',verified_name)
assert OUTPUT_DIR!=REPO_DIR and not OUTPUT_DIR.is_relative_to(REPO_DIR)
assert OUTPUT_DIR!=INPUT_DIR and not OUTPUT_DIR.is_relative_to(INPUT_DIR)
if OUTPUT_DIR.exists():raise FileExistsError('Output already exists; choose a new DUONG_ERROR_RESULTS_DIR, never overwrite prior results')
for p in [PREDICTIONS,AUDIT]:
    if not p.is_file():raise FileNotFoundError(f'Required real-data input is missing: {p}. No synthetic clinical substitutes are generated.')
input_paths={'predictions':PREDICTIONS,'patient_split_audit':AUDIT,'source_notebook':source_notebook}
if MANIFEST.is_file():input_paths['source_manifest']=MANIFEST
input_hashes={key:file_sha256(path) for key,path in input_paths.items()}


## 1. Validate every record before analysis
Required columns: model, target, seed, row_index, filename, patientid, y_true, y_pred. Require the exact three pipeline identifiers, two targets and all 50 seeds (0–49) for every model/target, unique prediction keys, finite scores, stable filename/patient/row identity and true labels, and complete model pairing. Source patient IDs are read as strings to preserve their CSV spelling, not renumbered or silently normalized.

The prior `patient_split_audit.json` is mandatory: validate every observed test index against its seed's actual audit, for both targets and all models; validate patient separation from train/test IDs (or derive IDs from the available full source manifest). Images are not expected to appear in every split. Missing entire paired observations are errors, not reasons to silently drop rows. Check ordered cohort/labels and 94 images/65 patients when the source manifest is supplied. A missing manifest is recorded explicitly; it is not replaced with invented provenance. Audit and manifest checks establish consistency of supplied artifacts, not independent certification of model training or checkpoint provenance.


In [ ]:
records=pd.read_csv(PREDICTIONS,dtype={'filename':str,'patientid':str})
audit=json.loads(AUDIT.read_text())
source_manifest=json.loads(MANIFEST.read_text()) if MANIFEST.is_file() else None
validated,cohort,validation=validate_predictions(records,audit,source_manifest)
print(json.dumps(validation,indent=2))
if source_manifest is None:print('Source reproducibility manifest unavailable: cohort-source identity remains incompletely verified.')
# Do not display raw source identities in presentation-ready notebook output.
display(presentation_table(validated).head())


## 2. Estimands: predictions → images → patients
For matched `(target, seed, row_index)` observations, error = prediction − truth, absolute error = |error| and squared error = error². Compute EVA minus comparator absolute error **within each matched observation first**; negative favours EVA.

For each unique image/target/model, average only occasions actually held out. Report n appearances, mean signed error, mean/median absolute error, sample SD of signed error (N/A for one appearance), mean prediction and true score. Mean absolute error is not the absolute error of the mean prediction.

For each patient, first obtain each image's mean held-out errors, then average distinct-image means with equal image weights. Images with more test appearances do not get extra patient weight. Patient paired differences also average image paired differences equally. Summaries report total scored images, analyzed images and per-image appearance counts. Patients with no observed test predictions are not assigned zero errors; coverage is reported. The lower-EVA-error proportion uses analyzed patients as its denominator, with ties reported separately.

These summaries remove repeated appearances from the plotted unit counts, but shared patients/training sets across splits still create dependence. No naive significance test or independent-replication standard error is computed.


In [ ]:
tables=analyze_predictions(validated,cohort)
display(presentation_table(tables['image_level_errors']).head())
display(presentation_table(tables['patient_level_errors']).head())
display(tables['patient_improvement_summary'])
display(presentation_table(tables['image_rankings']).head(12))


## 3. Bias and descriptive severity strata
Bias tables have two explicit weightings: (a) one unit per image's mean signed error, (b) one unit per equal-image patient mean signed error. Over/under rates first count each image’s held-out prediction fractions >0/<0, then average equally across distinct images (or equal-image patient summaries). Thus neither images with more appearances nor patients with more images dominate their respective weighting. Fractions of unit-mean signed errors >0/<0 and exact-zero means are reported separately; cancellation is not called low error. Mean/median signed error and mean absolute error answer different questions; cancellation of positive and negative errors does not indicate low MAE. No causal or clinical fairness assessment is implied.

Prespecified score strata are equal thirds of each allowed reference-score range: Geographic Extent `[0,8/3]`, `(8/3,16/3]`, `(16/3,8]`; Opacity `[0,2]`, `(2,4]`, `(4,6]`. These are descriptive score strata, **not validated clinical categories**. Each distinct image contributes once after resampling aggregation. A patient with several images may span strata; patient counts across strata are not disjoint. Each model/target/stratum reports distinct-image and patient counts, mean absolute and signed errors. Empty strata remain N/A. Groups below five images **or** five patients are flagged as small; do not draw robust subgroup conclusions from them. Trend lines connect these fixed-stratum means and fit no predictive model.


In [ ]:
display(tables['signed_error_summary'])
display(tables['severity_stratified_summary'])


## 4. Deliberately selected illustrative cases and privacy
Select one case per target/criterion with ties broken by smallest original row index:

- A: most negative image mean matched EVA−DenseNet absolute-error difference (strict improvement).
- B: largest positive difference (strict deterioration).
- C: largest positive EVA image mean signed error.
- D: most negative EVA image mean signed error.
- E: all three image mean absolute errors ≥1 score unit, maximize the smallest of the three MAEs. This threshold is descriptive, not clinical.

Cases may satisfy multiple criteria and are not representative performance samples. No eligible case means an explicit unavailable row, never a substitute. Case tables retain **mean signed and mean absolute errors separately**, true score, mean held-out prediction, count and both paired differences for all pipelines. Only out-of-sample predictions enter selection. No raw X-ray panels are attempted; a numerical comparison does not demonstrate pathological localization.

Presentation exports use `C###` image IDs and `P###` study patient IDs. Source filenames and patient IDs are confined to `private_audit/identity_mapping.csv` and `private_audit/representative_cases.csv` (plus private source manifest). These pseudonyms are study labels, not a guarantee of anonymization; do not publish the mapping or private folder. Never include original clinical images or direct identifiers in Git commits.


In [ ]:
display(presentation_table(tables['representative_cases']))


## 5. Write a new independent result set
Exports include all requested CSVs, six figures, supplementary image rankings and patient-improvement counts, source hashes, analysis versions, explicit estimands and selection/stratum rules. The original source manifest is kept privately; the public error-analysis manifest stores its hash and selected model provenance without repeating source identities. Input hashes record the exact artifacts used, not independent proof that the upstream pipeline was leakage-free.

No prior folder is overwritten. To rerun, choose a new explicit output directory (for example `/content/duong-error-analysis-results/run2`) before executing configuration. Do not delete or modify the earlier model results to make validation pass.


In [ ]:
for key,path in input_paths.items():
    if file_sha256(path)!=input_hashes[key]:raise RuntimeError('Input changed during analysis; stop and reload consistent artifacts')
analysis_manifest=export_analysis(tables,cohort,validation,OUTPUT_DIR,input_paths,source_manifest,synthetic_fixture=False)
print('New results saved to:',OUTPUT_DIR)
for name in FIGURE_NAMES:display(Image(filename=str(OUTPUT_DIR/name)))


## Lecture 9: critical assessment and limits
**Appropriate metrics:** signed error identifies direction, MAE captures magnitude without cancellation, squared error exposes large residuals, paired differences compare identical held-out observations, and hierarchical averaging makes the unit and weighting explicit. No percentage metric with a zero-denominator problem is introduced for zero scores.

**Interpretability:** error-vs-score plots and deliberately selected numerical cases explain observable failure patterns, not model attention, lesion localization or causation. Selection extremes and small strata are labelled. No fourth model, feature extraction, parameter refit or hyperparameter tuning occurs.

**Reproducibility:** source-export inspection, strict paired identity/membership checks, input hashes, original provenance, declared thresholds/tie-breaking, package versions and private mappings support auditability. All notebook outputs are empty until actual files pass validation. No clinical findings are inferred from synthetic fixtures or supplied summary means.

**Generalizability:** 94 images from 65 patients, annotation uncertainty, convenience sampling, possible upstream pretraining overlap and shared training sets constrain interpretation. Results are descriptive for these repeated internal splits; even broad patient-level improvement would not establish external validity, independent-patient inference, clinical benefit or statistical significance. Differences may reflect complete pretrained pipelines and data composition. Causal, fairness, localization and deployment claims require additional evidence.

**Execution status:** real prediction artifacts are absent in the development workspace. Only unit/synthetic integration tests can be executed there. No real patient rankings, improvement proportions, bias conclusions or study figures are supplied until this notebook runs on the real CSV and audit.
